# Chapter 1: Data Collection - Bitcointalk Profile Scraper

This notebook systematically scrapes user profiles from the Bitcointalk forum to extract public Bitcoin addresses and self-reported geographic locations. This data forms the foundational dataset for heuristic clustering and time zone analysis.

**Methodology:**
- Iterates through Bitcointalk profile IDs.
- Utilizes rotating proxies and randomized User-Agent headers to ensure reliable extraction and prevent rate-limiting.
- Periodically commits scraped data into a local MySQL database in batches to preserve memory and secure progress.

In [14]:
import os
import random
import subprocess
import time

import pandas as pd
import pymysql
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv

### 1. Database Configuration
Establish a connection to the local MySQL database where the scraped profile data will be stored securely. Credentials are loaded dynamically to avoid exposing passwords in the repository.

In [15]:
# Load environment variables (ensure your .env file is configured correctly)
load_dotenv()

# Establish connection to the MySQL server
# Note: Database parameters must match your local setup for the connection to succeed
conn = pymysql.connect(
    host='127.0.0.1',
    user='root',
    password=os.getenv("PASSWORD_MY_SQL"),
    database='data'
)
cursor = conn.cursor()

### 2. Web Scraping Configuration
Define the target URL structure and a robust list of rotating headers. This simulates requests from various operating systems and browsers, which helps bypass basic bot-protection mechanisms.

In [16]:
# Base URL representing the profile page. The {} acts as a placeholder for the User ID.
BASE_URL = "https://bitcointalk.org/index.php?action=profile;u={}" 

# A collection of diverse User-Agent headers to rotate during requests
HEADERS_LIST = [
    {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                      "AppleWebKit/537.36 (KHTML, like Gecko) "
                      "Chrome/118.0.5993.118 Safari/537.36",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.9",
        "Connection": "keep-alive"
    },
    {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:118.0) "
                      "Gecko/20100101 Firefox/118.0",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.9",
        "Connection": "keep-alive"
    },
    {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                      "AppleWebKit/537.36 (KHTML, like Gecko) "
                      "Edge/118.0.0.0 Safari/537.36",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.9",
        "Connection": "keep-alive"
    },
    {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64; Trident/7.0; rv:11.0) like Gecko",
        "Accept": "text/html, application/xhtml+xml, */*",
        "Accept-Language": "en-US,en;q=0.9",
        "Connection": "keep-alive"
    },
    {
        "User-Agent": "Mozilla/5.0 (Windows NT 6.1; Win64; x64) AppleWebKit/537.36 "
                      "(KHTML, like Gecko) Chrome/112.0.5615.138 Safari/537.36",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.9",
        "Connection": "keep-alive"
    }
]

### 3. Parsing Functions & Proxy Initialization
These utility functions extract the specific fields from the raw HTML. We also securely load the proxy list required to distribute the network requests.

In [17]:
def extract_field(soup, field_name):
    """
    Extracts a specific field from the user profile HTML (e.g., Location or Bitcoin address).
    Returns None if the field exists but is empty, or if the field does not exist.
    """
    # Find all table rows in the parsed HTML
    rows = soup.find_all("tr")
    for row in rows:
        # Find all data cells (<td>) within the current row
        cells = row.find_all("td")

        # We expect exactly two cells for a valid Label:Value pair
        if len(cells) == 2:
            # Extract text from the first cell, strip whitespace, and remove colons
            label = cells[0].get_text(strip=True).replace(":", "")
            
            # Check if the extracted label matches the target field name (case-insensitive)
            if label.lower() == field_name.lower():
                value = cells[1].get_text(strip=True)
                return value if value else None
                
    return None


def get_profile_info(user_id, res):
    """
    Parses the HTTP response of a user profile to extract their Bitcoin address and Location.
    Returns a standardized list formatted for database insertion.
    """
    # Parse the raw HTML content from the response
    soup = BeautifulSoup(res.text, "html.parser")

    # Attempt to extract the 'Location' and 'Bitcoin address' fields
    location = extract_field(soup, "Location")
    bitcoin_address = extract_field(soup, "Bitcoin address")

    # If the user has not provided a Bitcoin address, we skip saving the location
    if bitcoin_address == None:
        row = [user_id, None, None]
        return row

    location = location if location else None
    bitcoin_address = bitcoin_address if bitcoin_address else None
    
    # Log successful extractions to the console
    print(f"User ID: {user_id}")
    print(f"  Location: {location}")
    print(f"  Bitcoin Address: {bitcoin_address}")
    print("-" * 50)
    
    row = [user_id, bitcoin_address, location]
    return row


# Define the file path for the proxy list via the environment variable
path = os.getenv("RAW_DATA_PATH")
proxies_dict_list = []

# Open the text file containing the proxies in read mode with UTF-8 encoding
with open(path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        
        # Split the TXT format IP:PORT:USER:PASS
        ip, port, user, pwd = line.split(":")
        
        # Create a dictionary configured directly for the requests library
        proxy_dict = {
            "http": f"http://{user}:{pwd}@{ip}:{port}",
            "https": f"http://{user}:{pwd}@{ip}:{port}"
        }

        proxies_dict_list.append(proxy_dict)

### 4. Main Scraping Execution
The primary data collection loop. Iterates over a defined range of profile IDs, utilizing retry logic to handle network interruptions. Data is written to the database iteratively in batches of 1,000 to maintain stability over long scraping sessions.

In [18]:
# Initialize timing and define the batch boundaries for the current scraping session
time1 = time.time()
start_id = 0
end_id = 100000
i = 0

data = []
numbers = list(range(start_id, end_id + 1))

for uid in numbers:
    url = BASE_URL.format(uid)
    
    # Attempt to fetch data up to three times per URL to handle potential timeouts.
    # Each attempt selects a random proxy and a random User-Agent header.
    try:
        res = requests.get(url, proxies=random.choice(proxies_dict_list), headers=random.choice(HEADERS_LIST), timeout=10)
    except requests.exceptions.RequestException:
        print("error")
        time.sleep(5)
        try:
            res = requests.get(url, proxies=random.choice(proxies_dict_list), headers=random.choice(HEADERS_LIST), timeout=10)
        except requests.exceptions.RequestException:
            print("error")
            time.sleep(5)
            res = requests.get(url, proxies=random.choice(proxies_dict_list), headers=random.choice(HEADERS_LIST), timeout=10)
    
    # Check for HTTP access errors (e.g., 403 Forbidden or 429 Too Many Requests)
    if res.status_code != 200:
        print(f"User {uid}: not reachable")
        # Retry one final time upon receiving a bad status code
        res = requests.get(url, proxies=random.choice(proxies_dict_list), headers=random.choice(HEADERS_LIST), timeout=10)

    # Parse the HTML and extract the required user profile data
    row = get_profile_info(uid, res)
    data.append(row)
    i += 1

    # Batch Process: Once every 1,000 scraped users, insert the data block into the MySQL database
    if i % 1000 == 0:
        cursor.executemany("""INSERT IGNORE INTO scraped_data (id, address, location) VALUES (%s, %s, %s)""", data)
        conn.commit()
        
        # Reset the data list for the next batch and log the time elapsed
        data = []
        print(time.time() - time1)

User ID: 30
  Location: Un-United Kingdom
  Bitcoin Address: 1JXFXUBGs2ZtEDAQMdZ3tkCKo38nT2XSEp
--------------------------------------------------


KeyboardInterrupt: 